## How Objects Access Attributes & Dynamic Creation

### How Attribute Access Works
When you type `obj.name`, Python follows a specific lookup order:
1. It checks the **Instance Namespace** (stored in `obj.__dict__`).
2. If not found, it checks the **Class Namespace** (for static variables/methods).
3. If still not found, it throws an `AttributeError`.

### Creating Attributes from Outside the Class
Python is dynamically typed. You can add new attributes to an object *after* it has been created.
* **Warning:** While Python allows this, it is considered **bad practice** in production code because it breaks structural uniformity. If you add `student1.age = 20` outside the class, `student2` will not have an `age` attribute, leading to crashes.


In [2]:
# ATTRIBUTE ACCESS & DYNAMIC CREATION
class Employee:
    def __init__(self, emp_id, name):
        self.emp_id = emp_id
        self.name = name

e1 = Employee(1, "John")
e2 = Employee(2, "Jane")

# 1. Viewing the internal dictionary (Instance Namespace)
print(f"e1 internal state: {e1.__dict__}")

# 2. Safe Attribute Access using built-in functions
print(f"\ngetattr (safe): {getattr(e1, 'name', 'Unknown')}")
print(f"getattr (default): {getattr(e1, 'age', 'Not Found')}")
print(f"hasattr (check): {hasattr(e1, 'emp_id')}")

# 3. Dynamic Attribute Creation (Outside the class)
e1.department = "IT"       # Added ONLY to e1
print(f"\nDynamically added: {e1.department}")

# print(e2.department)  # ❌ AttributeError! e2 doesn't have this attribute.
print(f"e2 internal state: {e2.__dict__}") # Notice 'department' is missing here


e1 internal state: {'emp_id': 1, 'name': 'John'}

getattr (safe): John
getattr (default): Not Found
hasattr (check): True

Dynamically added: IT
e2 internal state: {'emp_id': 2, 'name': 'Jane'}


## Reference Variables & Memory

### What is a Reference Variable?
In Python, **variables are NOT buckets that hold data; they are name tags (references) that point to objects in memory.**
* When you write `a = [1, 2]`, Python creates a list object in the Heap memory and attaches the name tag `a` (stored in the Stack) to it.
* If you write `b = a`, both `a` and `b` point to the **exact same object** in memory.
* We can create objects without reference variable as well
* An object can have multiple reference variable.
* Assigning a new reference variable to an existing object doesn't create a new object.

### `is` vs `==`
* `==` checks for **Value Equality** (Do they contain the same data?).
* `is` checks for **Reference Equality** (Do they point to the exact same memory address?).


In [10]:
# 1. Reference Variables
list1 = [1, 2, 3]
list2 = list1  # list2 points to the SAME memory address as list1

print(f"ID of list1: {id(list1)}")
print(f"ID of list2: {id(list2)}")
print(f"Are they the same object? {list1 is list2}")  # True!

# Modifying through one reference affects the other!
list2.append(4)
print(f"\nlist1 after modifying list2: {list1}") 

# 2. Integer (Immutable) Reference Behavior
x = 100
y = 100
print(f"\nAre x and y the same object? {x is y}") # True (Python caches small integers)

y = 200 # y now points to a NEW integer object
print(f"After y=200, are they same? {x is y}") # False

ID of list1: 2186135543552
ID of list2: 2186135543552
Are they the same object? True

list1 after modifying list2: [1, 2, 3, 4]

Are x and y the same object? True
After y=200, are they same? False


In [18]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks
        
    def get_status(self):
        return "Pass" if self.marks >= 40 else "Fail"
        
    def __str__(self):
        return f"{self.name} ({self.marks} marks)"

# =====================================================================
# USE CASE 1: Calling a method directly on an anonymous object
# =====================================================================
# We create the object and immediately call get_status() 
# No variable like `s1 = Student(...)` is needed.
status = Student("Alice", 85).get_status()
print(f"1. Direct method call: {status}")


1. Direct method call: Pass


In [19]:
# =====================================================================
# USE CASE 2: Passing an anonymous object to a function
# =====================================================================
def process_student(student_obj):
    print(f"2. Passed to function: Processing {student_obj.name} with marks {student_obj.marks}")

# The object is created directly inside the function call
process_student(Student("Bob", 92))

2. Passed to function: Processing Bob with marks 92


In [20]:
# =====================================================================
# USE CASE 3: Storing anonymous objects in a collection (List/Dict)
# =====================================================================
# Useful when you have a batch of objects but don't need to reference 
# them individually by name later.
students_list = [
    Student("Charlie", 35),  # Anonymous object added to list
    Student("Diana", 78),    # Anonymous object added to list
    Student("Evan", 50)      # Anonymous object added to list
]

print("3. Anonymous objects in a list:")
for s in students_list:
    print(f"   - {s.name}: {s.get_status()}")

3. Anonymous objects in a list:
   - Charlie: Fail
   - Diana: Pass
   - Evan: Pass


### 🧠 What happens to them in memory?
Even though you don't give the object a name, **Python still creates it in memory**. 
* If you call a method on it directly (Use Case 1), Python executes the method and then immediately destroys the object (Garbage Collection) because its reference count drops to zero.
* If you pass it to a function (Use Case 2), the function's parameter temporarily holds the reference. Once the function finishes executing, the object is destroyed.
* If you add it to a list (Use Case 3), the **list** holds the reference, so the object stays in memory as long as the list exists.

### 💡 When should you use it?
* **✅ DO USE** when the object's lifespan is limited to a single operation (like a quick math calculation, a one-time API request wrapper, or adding to a database/collection).
* **❌ DO NOT USE** if you need to access the object's state multiple times throughout your code. In that case, assigning it to a reference variable (`my_obj = MyClass()`) is much cleaner and more readable.

## 🎯 Interview Question

**Q: "What happens to an object without a reference variable?"**

**A:** The object is created and can be used immediately, but since no variable stores its memory address, it becomes **unreachable** after the statement completes. Python's **Garbage Collector** automatically reclaims the memory. This is useful for one-time operations but wasteful if the same object is needed multiple times.

## Pass by Object Reference & Mutability

### Python's "Pass by Object Reference"
*Note: Python is strictly **Pass by Value**, but the "value" being passed is a **reference** to the object.*
* **Rule 1 (Immutable):** If you pass an immutable object (int, str, tuple) and try to change it, Python creates a **new local object**. The original remains unchanged.
* **Rule 2 (Mutable):** If you pass a mutable object (list, dict, custom object) and **mutate** it (e.g., `param.append()`), it **will** affect the original object.
* **Rule 3 (Reassignment):** If you **reassign** the parameter inside the function (e.g., `param = new_value`), it does **not** affect the original object, regardless of mutability.

In [24]:
# 1. Class Definition
class Person: 
    def __init__(self, name, gender):
        self.name = name
        self.gender = gender

# 2. Standalone Function (Not a method, because it is outside the class)
def greet(person):
    # 'person' is a reference variable pointing to the object we passed in (Anuj)
    print(f"Hello my name is {person.name}, and I'm a {person.gender}")
    
    # We can create a brand new object inside a function
    otherPerson = Person("Niketan", "Male")
    
    # We return the reference to this new object
    return otherPerson

# 3. Execution
p = Person("Anuj", "Male")

# We pass 'p' into the function. 
# The function catches it, prints Anuj's info, creates Niketan, and returns Niketan.
# 'x' catches the returned Niketan object.
x = greet(p)

# We print the attributes of the object that 'x' is holding
print(x.name, x.gender)

Hello my name is Anuj, and I'm a Male
Niketan Male


### 🔍 Code Analysis: Passing and Returning Objects

This code snippet is an excellent demonstration of how Python handles custom objects when interacting with standalone functions. It specifically highlights two concepts:
1. **Passing an object as an argument** to a function (Pass by Object Reference).
2. **Returning a completely new object** from a function.

### 🧠 Step-by-Step Execution Trace

1. **`p = Person("Anuj", "Male")`**: Python creates a `Person` object in memory with the name "Anuj". The reference variable `p` points to it.
2. **`greet(p)`**: You call the `greet` function and hand it the reference `p`. 
3. **Inside `greet(person)`**: 
   * The local variable `person` now points to the exact same "Anuj" object.
   * `print(f"Hello...")` runs, accessing `person.name` ("Anuj") and `person.gender` ("Male").
   * `otherPerson = Person("Niketan", "Male")` creates a **second, brand new object** in memory.
   * `return otherPerson` spits out the reference to this new "Niketan" object.
4. **`x = greet(p)`**: The variable `x` catches the returned reference. Now, `x` points to the "Niketan" object.
5. **`print(x.name, x.gender)`**: Python looks at the object `x` is pointing to and prints "Niketan Male".

In [ ]:
# MUTABILITY & PASS BY REFERENCE

def modify_data(my_list, my_number):
    # Mutating a mutable object (List) -> Affects original!
    my_list.append(99) 
    
    # Reassigning an immutable object (Int) -> Creates new local reference!
    my_number = 500 

original_list = [1, 2, 3]
original_num = 100

modify_data(original_list, original_num)

print(f"After function call:")
print(f"List: {original_list}")   # [1, 2, 3, 99] -> CHANGED!
print(f"Number: {original_num}")  # 100 -> UNCHANGED!

# --- Mutability Demonstration ---
s1 = "Hello"
print(f"\nID of s1 before: {id(s1)}")
s1 = s1 + " World"  # Creates a NEW string object
print(f"ID of s1 after:  {id(s1)}")  # ID changes! Strings are IMMUTABLE.


After function call:
List: [1, 2, 3, 99]
Number: 100

ID of s1 before: 2186135625936
ID of s1 after:  2186135575664


## Encapsulation & Pythonic Getters/Setters

### What is Encapsulation?
Encapsulation is the OOP principle of bundling data (attributes) and the methods that operate on that data into a single unit (the Class), while **restricting direct access** to some of the object's components.

### The Real-World Analogy
> Think of a **Smartphone**. 
> You interact with the screen, the volume buttons, and the charging port (The **Public Interface**). You are *not* allowed to directly touch the battery, the motherboard, or the CPU (The **Private Data**). If you want to increase the battery level (modify data), you must plug it into the charger (a **Setter Method**). This prevents you from accidentally electrocuting yourself or destroying the phone.

### Why do we need it?
1. **Security:** Prevents unauthorized modification of sensitive data (e.g., changing a bank balance directly).
2. **Validation:** Allows you to check rules before changing data (e.g., an age cannot be negative).
3. **Flexibility:** You can change how data is stored internally without breaking the code of the people using your class.




In [1]:
# --- THE PROBLEM: Without Encapsulation ---
class User:
    def __init__(self, username, password):
        self.username = username
        self.password = password # Public! Dangerous!

u1 = User("anuj_dev", "secret123")
u1.password = "hacked!" # Anyone can change this directly!
print(f"Password maliciously changed to: {u1.password}")

# --- THE SOLUTION: Private Variables ---
class SecureUser:
    def __init__(self, username, password):
        self.username = username
        # Prefix with double underscore (__) to make it private
        self.__password = password 

u2 = SecureUser("rahul_dev", "secure456")

# print(u2.__password) # ⚠️ AttributeError! Python hides this variable.
# u2.__password = "hacked" # This does NOT change the internal password, it creates a new dummy variable!

Password maliciously changed to: hacked!


---
##  Traditional Getters and Setters

Since we hid the data using private variables (`__`), how does the rest of the program interact with it? We build **Getters** and **Setters**.

* **Getter (Accessor):** A public method that returns the value of a private variable. It allows *Read-Only* access.
* **Setter (Mutator):** A public method that takes a parameter and assigns it to the private variable. **Crucially, this is where you put your validation logic (If/Else statements).**

In [2]:
class BankAccount:
    def __init__(self, owner_name, initial_balance):
        self.owner_name = owner_name
        # Private variable
        self.__balance = initial_balance 
        
    # --- GETTER METHOD ---
    def get_balance(self):
        """Allows users to securely view their balance."""
        return self.__balance
        
    # --- SETTER METHOD ---
    def set_balance(self, new_amount):
        """Allows users to update their balance, BUT ONLY IF it passes validation."""
        if type(new_amount) not in [int, float]:
            print("❌ Error: Balance must be a number.")
            return
            
        if new_amount < 0:
            print("❌ Error: Balance cannot be negative.")
            return
            
        # If it passes all security checks, update the private variable
        self.__balance = new_amount
        print(f"✅ Balance successfully updated to ₹{self.__balance}")

# Testing Traditional Getters and Setters
account = BankAccount("Anuj", 5000)

# Reading data safely
print(f"Current Balance: ₹{account.get_balance()}")

# Attempting bad modifications (Blocked by Setter!)
account.set_balance("One Million") # Fails type check
account.set_balance(-500)          # Fails negative check

# Successful modification
account.set_balance(15000)

Current Balance: ₹5000
❌ Error: Balance must be a number.
❌ Error: Balance cannot be negative.
✅ Balance successfully updated to ₹15000


### Access Modifiers in Python:
1. **Public:** `self.name` (Accessible everywhere).
2. **Protected:** `self._name` (Convention only. Tells devs "internal use").
3. **Private:** `self.__name` (Strictly enforced via **Name Mangling**. Cannot be accessed directly using `obj.__name`).

### The Pythonic Way: `@property`
While traditional `get_balance()` and `set_balance()` work (and are how Java/C++ do it), Python developers consider them "ugly" and un-Pythonic. 

Python provides the `@property` decorator. This allows you to define getter and setter methods, but the user can use them exactly like regular variables (using the `=` sign). It gives you **the syntax of public variables, with the security of private methods.**

In [3]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.__salary = salary # Private
        
    # 1. THE GETTER (Using @property)
    # This turns the method into an attribute. 
    @property
    def salary(self):
        """Gets the salary safely."""
        return self.__salary
        
    # 2. THE SETTER (Using @attribute_name.setter)
    @salary.setter
    def salary(self, new_salary):
        """Sets the salary with validation."""
        if new_salary < 30000:
            raise ValueError("❌ Minimum wage law violated! Salary must be at least 30,000.")
        self.__salary = new_salary


# Testing the Pythonic Way
emp = Employee("Priya", 50000)

# Looks like we are accessing a public variable, but it secretly runs the Getter method!
print(f"Initial Salary: ₹{emp.salary}")

# Looks like a direct assignment, but it secretly runs the Setter method!
emp.salary = 80000
print(f"Promoted Salary: ₹{emp.salary}")

# This will trigger the ValueError in the setter!
try:
    emp.salary = 10000 
except ValueError as e:
    print(e)

Initial Salary: ₹50000
Promoted Salary: ₹80000
❌ Minimum wage law violated! Salary must be at least 30,000.


In [27]:
# 🔒 ENCAPSULATION & @PROPERTY DECORATORS

class Temperature:
    def __init__(self, celsius):
        self.__celsius = celsius  # Private variable
        
    # GETTER: Allows access like an attribute (temp.celsius)
    @property
    def celsius(self):
        return self.__celsius
        
    # SETTER: Allows validation when assigning (temp.celsius = 30)
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature cannot be below absolute zero!")
        self.__celsius = value
        
    @property
    def fahrenheit(self):
        return (self.__celsius * 9/5) + 32

# Testing Encapsulation
temp = Temperature(25)

print(f"Celsius: {temp.celsius}°C")      # Uses @property getter
print(f"Fahrenheit: {temp.fahrenheit}°F") # Computed on the fly

Celsius: 25°C
Fahrenheit: 77.0°F


In [28]:
temp.celsius = 30  # Uses @celsius.setter
print(f"Updated Celsius: {temp.celsius}°C")

Updated Celsius: 30°C


In [29]:
# print(temp.__celsius) # ❌ AttributeError! Name mangling hides it.
# Access via mangling (BAD PRACTICE):
print(f"Hidden value via mangling: {temp._Temperature__celsius}") 


Hidden value via mangling: 30


---
##  Top Interview Questions (Encapsulation)

1.  **"What is Encapsulation and what problem does it solve?"**
    * *Answer:* Encapsulation is the bundling of data and methods within a class, and restricting direct external access to the data. It solves the problem of accidental or malicious data modification by forcing users to interact with the data through controlled, validated methods (Getters/Setters).
2.  **"How do you make an attribute read-only in Python?"**
    * *Answer:* You make the attribute private (e.g., `self.__data`), and you provide a Getter method (or an `@property` decorator) to return its value, but you **do not** write a Setter method. Without a setter, the outside world can read it, but never change it.
3.  **"Does Python have true, unbreakable private variables?"**
    * *Answer:* No. Python uses a technique called **Name Mangling**. When you type `self.__balance`, Python internally renames it to `_ClassName__balance`. A developer who knows this rule can still bypass it by typing `obj._BankAccount__balance`. Python relies on the philosophy that "we are all consenting adults"—it prevents *accidental* modification, but not *intentional* hacking.
4.  **"Why should we use the `@property` decorator instead of `get_value()` and `set_value()` methods?"**
    * *Answer:* The `@property` decorator is more "Pythonic." It allows developers to use simple, clean assignment syntax (e.g., `obj.value = 10`) while still secretly running the validation logic hidden inside the setter method under the hood. It keeps the public API clean and intuitive.

## Collection of Objects

In real-world applications, we rarely deal with just one object. We manage **collections (lists, dictionaries) of objects**. This allows us to perform bulk operations, filtering, and sorting using Python's powerful built-in functions and list comprehensions.


In [30]:
# COLLECTION OF OBJECTS (Library System)

class Book:
    def __init__(self, title, price, category):
        self.title = title
        self.price = price
        self.category = category
        
    def __repr__(self):
        return f"'{self.title}' (${self.price})"

inventory = [
    Book("Python Basics", 20, "Tech"),
    Book("Data Science", 50, "Tech"),
    Book("The Hobbit", 15, "Fiction"),
    Book("AI Revolution", 60, "Tech"),
    Book("Dune", 18, "Fiction")
]

# 1. Filtering (Find all Tech books)
tech_books = [b for b in inventory if b.category == "Tech"]
print(f"Tech Books: {tech_books}")

Tech Books: ['Python Basics' ($20), 'Data Science' ($50), 'AI Revolution' ($60)]


In [31]:
# 2. Sorting (Find most expensive)
most_expensive = max(inventory, key=lambda b: b.price)
print(f"Most Expensive: {most_expensive}")

Most Expensive: 'AI Revolution' ($60)


In [32]:
# 3. Aggregation (Total inventory value)
total_value = sum(b.price for b in inventory)
print(f"Total Value: ${total_value}")


Total Value: $163


## Types of Methods

* **Python Method Ecosystem**
  * **1. Normal Methods (Instance Methods)**
    * The default method type.
    * Takes `self` as the first parameter.
    * Can access and modify the unique data of a specific object.
  * **2. Static Methods (`@staticmethod`)**
    * Takes neither `self` nor `cls`.
    * Cannot access or modify object data or class data.
    * Behaves exactly like a regular independent function, but placed inside a class for logical grouping.
  * **3. Utility Methods (The Use-Case)**
    * Not a special Python keyword, but a *design concept*.
    * These are usually implemented as **Static Methods**.
    * Used for generic tasks: validation, formatting, or calculations that are relevant to the class but don't need its data.
  * **4. Class Methods (`@classmethod` - Bonus)**
    * Takes `cls` as the first parameter. Used to modify class-level (static) variables or act as alternative constructors.

---

##  Theory & Real-World Analogy

Imagine a **Restaurant**:
1. **Normal (Instance) Method -> `take_order(self, food)`:** Every waiter (object) takes a different order for their specific table. The action depends entirely on the specific waiter and table (`self`).
2. **Static Method -> `calculate_tax(amount)`:** Calculating 5% tax is the exact same math formula everywhere. It doesn't matter which waiter does it, and it doesn't change the restaurant's name. It's just a generic math rule placed inside the Restaurant class.
3. **Utility Method -> `validate_credit_card(card_number)`:** A specific type of Static Method. Its pure utility is to check if a 16-digit number is valid before processing payment.

### Why not just write functions outside the class?
You *could* write `validate_credit_card()` outside the class as a normal function. However, placing it inside the `Restaurant` class as a `@staticmethod` keeps your code **organized and encapsulated**. It tells other developers: *"This validation logic specifically belongs to the restaurant's payment domain."*

In [12]:
# --- DEMONSTRATING METHOD TYPES ---

class Employee:
    company_name = "TechCorp" # Class Variable

    def __init__(self, name, salary, email):
        self.name = name       # Instance Variable
        self.salary = salary   # Instance Variable
        self.email = email     # Instance Variable

    # ---------------------------------------------------------
    # 1. NORMAL METHOD (Instance Method)
    # ---------------------------------------------------------
    # Takes 'self'. Operates on the unique data of the object.
    def apply_raise(self, percentage):
        """Modifies the specific employee's salary."""
        raise_amount = self.salary * (percentage / 100)
        self.salary += raise_amount
        return f"✅ {self.name} received a raise! New Salary: ₹{self.salary}"

    def get_details(self):
        """Reads the specific employee's data."""
        return f"Name: {self.name} | Email: {self.email} | Salary: ₹{self.salary}"

    # ---------------------------------------------------------
    # 2. CLASS METHOD (Bonus Concept for Completeness)
    # ---------------------------------------------------------
    # Takes 'cls'. Modifies class-level data.
    @classmethod
    def change_company_name(cls, new_name):
        cls.company_name = new_name
        return f"Company renamed to {cls.company_name}"

    # ---------------------------------------------------------
    # 3. STATIC / UTILITY METHOD
    # ---------------------------------------------------------
    # Takes NO implicit first argument (no self, no cls).
    # Acts as a pure utility function grouped inside the class.
    @staticmethod
    def is_valid_email(email_address):
        """Utility: Validates if a string looks like an email."""
        if "@" in email_address and "." in email_address:
            return True
        return False
        
    @staticmethod
    def format_currency(amount):
        """Utility: Formats a raw number into a readable currency string."""
        return f"₹{amount:,.2f}"


# ==========================================
# TESTING THE METHODS
# ==========================================

print("--- 1. Testing Utility / Static Methods ---")
# Notice we call these DIRECTLY on the Class, without needing to create an object!
test_email1 = "anuj@gmail.com"
test_email2 = "invalid_email.com"

print(f"Is '{test_email1}' valid? {Employee.is_valid_email(test_email1)}")
print(f"Is '{test_email2}' valid? {Employee.is_valid_email(test_email2)}")
print(f"Formatted amount: {Employee.format_currency(500000)}")

print("\n--- 2. Testing Normal (Instance) Methods ---")
# We MUST create an object to use Instance Methods
emp1 = Employee("Anuj", 50000, "anuj@gmail.com")

print(emp1.get_details())
print(emp1.apply_raise(10)) # Anuj gets a 10% raise

--- 1. Testing Utility / Static Methods ---
Is 'anuj@gmail.com' valid? True
Is 'invalid_email.com' valid? False
Formatted amount: ₹500,000.00

--- 2. Testing Normal (Instance) Methods ---
Name: Anuj | Email: anuj@gmail.com | Salary: ₹50000
✅ Anuj received a raise! New Salary: ₹55000.0


---
##  Deep Dive: When to use which?

### The Decision Tree:
When writing a method inside a class, ask yourself this question:
* **Question 1:** *Does this method need to access or modify `self.name` or `self.balance` (Instance Variables)?*
  * **Yes ->** Use a **Normal (Instance) Method**.
  * **No ->** Go to Question 2.
* **Question 2:** *Does this method need to access or modify `ClassName.total_count` (Class Variables)?*
  * **Yes ->** Use a **Class Method** (`@classmethod`).
  * **No ->** It doesn't need the object, and it doesn't need the class. It just uses the arguments passed to it. **Use a Static/Utility Method** (`@staticmethod`).

### The Role of Utility Methods
A **Utility Method** is just a descriptive name for a specific type of Static Method. 
They are typically used for:
1. **Data Validation:** Checking if a phone number has 10 digits before creating a `User` object.
2. **Formatting:** Converting a raw timestamp into a readable date string.
3. **Mathematical Calculations:** Calculating compound interest given a rate and time.

In [ ]:
#  STATIC VARIABLES, CLASS METHODS & STATIC METHODS

class Validator:
    """A class consisting entirely of Static Methods (Utility Functions)"""
    
    @staticmethod
    def is_valid_email(email):
        return "@" in email and "." in email
        
    @staticmethod
    def is_adult(age):
        return age >= 18

class Employee:
    company_name = "TechCorp"  # Static Variable
    
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary
        
    @classmethod
    def change_company(cls, new_name):
        cls.company_name = new_name
        
    @staticmethod
    def calculate_bonus(salary):
        return salary * 0.10  # Pure math, doesn't need self or cls

# Testing Static Methods (No object needed!)
print(f"Email valid? {Validator.is_valid_email('test@mail.com')}")
print(f"Bonus: ${Employee.calculate_bonus(50000)}")


Email valid? True
Bonus: $5000.0


---
## Top Interview Questions (Methods)

1.  **"What is the difference between an Instance Method and a Static Method?"**
    * *Answer:* An instance method takes `self` as its first parameter and can access and modify the internal state of the specific object it is called on. A static method uses the `@staticmethod` decorator, does not take `self` or `cls`, and cannot modify object or class state. It acts like a standard standalone function but is kept inside the class for logical organization.
2.  **"Can a Static Method call an Instance Method?"**
    * *Answer:* Directly, no. Because a static method does not have the `self` reference, it does not know *which* object's instance method to call. If you want a static method to call an instance method, you must explicitly pass an object instance into the static method as an argument.
3.  **"Why would you use a `@staticmethod` instead of just writing a regular function outside the class?"**
    * *Answer:* For namespace organization and encapsulation. If a utility function (like `is_valid_email`) is only ever going to be used in the context of creating an `Employee`, placing it inside the `Employee` class makes the codebase cleaner, easier to navigate, and groups logically related code together.
4.  **"What is a Utility Method in Python?"**
    * *Answer:* "Utility method" is not a native Python keyword, but rather a design pattern. It refers to a method that performs a generic, repetitive task (like formatting a string or validating an input). In Python, utility methods are almost always implemented using the `@staticmethod` decorator because they rely entirely on the parameters passed to them rather than the state of the object.

In [5]:
# Testing Static Variables & Class Methods
e1 = Employee("Alice", 80000)
print(f"\nCompany: {e1.company_name}")

Employee.change_company("InnovateX")
print(f"Updated Company: {e1.company_name}") # Changed for ALL instances!


Company: TechCorp
Updated Company: InnovateX


In [6]:
# ⚠️ THE SHADOWING TRAP
e1.company_name = "PersonalCorp" # ❌ Creates an INSTANCE variable!
print(f"e1 Company: {e1.company_name}")      # PersonalCorp (Instance)
print(f"Employee Company: {Employee.company_name}") # InnovateX (Class)


e1 Company: PersonalCorp
Employee Company: InnovateX


## 🎯 Top Interview Questions on Advanced OOP

**Q1: How does Python internally access object attributes?**
> **Answer:** Python stores instance attributes in a dictionary called `__dict__`. When you access `obj.attr`, Python checks `obj.__dict__`, then the class `__dict__`, and follows the MRO (Method Resolution Order) chain.

**Q2: Does Python use pass-by-value or pass-by-reference?**
> **Answer:** Python uses **"Pass by Object Reference"**. If you pass a mutable object (like a list) and mutate it, the original changes. If you pass an immutable object (like an int) or reassign the parameter, the original remains unchanged.

**Q3: What is the difference between Mutable and Immutable objects?**
> **Answer:** Mutable objects (list, dict, set, custom classes) can be changed in-place. Immutable objects (int, float, str, tuple) cannot be changed; any modification creates a brand new object in memory.

**Q4: How does Encapsulation work in Python? Are there true private variables?**
> **Answer:** Python doesn't have strict `private` keywords like Java. It uses **Name Mangling** (prefixing with `__`) to rename variables to `_ClassName__var`, preventing accidental access. The modern Pythonic way to encapsulate with validation is using the `@property` decorator.

**Q5: What happens if you assign a value to a Static Variable through an instance?**
> **Answer:** It creates a new **Instance Variable** with the same name, "shadowing" the class variable for that specific object. The original class variable remains unchanged for other instances.

**Q6: When should you use `@staticmethod` vs `@classmethod`?**
> **Answer:** Use `@classmethod` when you need to access or modify **Class Variables**, or as an alternative constructor. Use `@staticmethod` for pure utility/math functions that logically belong to the class but don't need to access any class or instance data.

* **Variables in OOP**
  * **1. Instance Variables (Object Level)**
    * Defined inside `__init__` using `self.variable_name`.
    * Unique to *each individual object*.
    * Stored in the object's personal dictionary (`__dict__`).
  * **2. Static/Class Variables (Class Level)**
    * Defined directly inside the class body, outside any methods.
    * Shared across *all objects* of that class.
    * Stored in the Class's namespace, not the object's.
  * **3. The "Shadowing" Trap**
    * What happens when an object tries to overwrite a class variable? (It creates a local copy instead).

---

## Theory: The Company Analogy

To understand the difference, imagine a software company:
* **Static Variable:** The `company_name` or `office_address`. Every employee shares the exact same company name. If the company rebrands, the name changes for *everyone* instantly. It belongs to the Company (the Class).
* **Instance Variable:** The `employee_name` or `salary`. Every employee has their own unique name and salary. If Employee A gets a raise, Employee B's salary does not change. It belongs to the Employee (the Object).

In [9]:
# --- DEMONSTRATING INSTANCE VS STATIC VARIABLES ---

class Employee:
    # 1. STATIC (CLASS) VARIABLES
    # Defined in the class body. Shared by ALL instances.
    company_name = "TechCorp Solutions"
    total_employees = 0 
    
    def __init__(self, name, salary):
        # 2. INSTANCE VARIABLES
        # Defined inside __init__ using 'self'. Unique to EACH instance.
        self.name = name
        self.salary = salary
        
        # We access the static variable using the ClassName to increment it
        Employee.total_employees += 1

    def display_info(self):
        # We can read static variables using 'self', but modifying them requires the ClassName
        print(f"[{self.company_name}] {self.name} earns ₹{self.salary}")


# --- Testing the Behavior ---
print("--- Initial State ---")
emp1 = Employee("Anuj", 50000)
emp2 = Employee("Priya", 60000)

emp1.display_info()
emp2.display_info()
print(f"Total Employees Count: {Employee.total_employees}")

--- Initial State ---
[TechCorp Solutions] Anuj earns ₹50000
[TechCorp Solutions] Priya earns ₹60000
Total Employees Count: 2


In [10]:
# --- Modifying Variables ---
print("\n--- After Modifications ---")
# 1. Modifying an Instance Variable (Only affects emp1)
emp1.salary = 55000 

# 2. Modifying a Static Variable via the Class (Affects EVERYONE)
Employee.company_name = "InnovateX Global"

emp1.display_info()
emp2.display_info() # Notice emp2's company name also changed!


--- After Modifications ---
[InnovateX Global] Anuj earns ₹55000
[InnovateX Global] Priya earns ₹60000


---
## The "Shadowing" Pitfall (Crucial Concept)

A very common mistake in Python OOP happens when you try to modify a Static (Class) Variable using an *Object Reference* (e.g., `obj.static_var = "new"`). 

**The Rule of Shadowing:**
* If you **read** a static variable using an object (`print(emp1.company_name)`), Python looks in the object's `__dict__`. If it doesn't find it, it looks up at the Class level and reads it successfully.
* If you **write/assign** to a static variable using an object (`emp1.company_name = "Apple"`), Python **does not** go up to the Class level. Instead, it creates a brand new Instance Variable inside `emp1` with the exact same name. This new instance variable "shadows" (hides) the class variable for that specific object.

In [11]:
class Car:
    # Static Variable
    wheels = 4 
    
    def __init__(self, brand):
        # Instance Variable
        self.brand = brand

c1 = Car("Honda")
c2 = Car("Toyota")

print(f"Initial State -> c1 wheels: {c1.wheels} | c2 wheels: {c2.wheels}")

# THE MISTAKE: Trying to change the static variable via an object
c1.wheels = 5 

print("\n--- After c1.wheels = 5 ---")
# c1 now has its own unique 'wheels' variable, but the class variable is untouched!
print(f"c1 wheels: {c1.wheels}") 
print(f"c2 wheels: {c2.wheels}") 
print(f"Class wheels: {Car.wheels}") 

print("\n--- Proof from Memory (__dict__) ---")
# Let's look inside the memory dictionaries
print(f"c2 memory: {c2.__dict__}") # Only has 'brand'
print(f"c1 memory: {c1.__dict__}") # Now has BOTH 'brand' and a local copy of 'wheels'!

# THE CORRECT WAY: Always use the Class Name to modify Static Variables
Car.wheels = 6
print(f"\nAfter Car.wheels = 6 -> c2 wheels: {c2.wheels}") # c2 updates because it still points to the class!

Initial State -> c1 wheels: 4 | c2 wheels: 4

--- After c1.wheels = 5 ---
c1 wheels: 5
c2 wheels: 4
Class wheels: 4

--- Proof from Memory (__dict__) ---
c2 memory: {'brand': 'Toyota'}
c1 memory: {'brand': 'Honda', 'wheels': 5}

After Car.wheels = 6 -> c2 wheels: 6


---
## Top Interview Questions (Variables)

1.  **"What is the difference between a Class Variable and an Instance Variable?"**
    * *Answer:* Instance variables are tied to a specific object and are created using `self` inside methods like `__init__`. Each object has its own separate copy. Class (Static) variables are declared inside the class body, outside of any methods. They are shared across all instances of the class, meaning there is only one copy in memory.
2.  **"How should you modify a static variable to ensure the change reflects across all objects?"**
    * *Answer:* You must modify it using the Class Name (e.g., `ClassName.static_variable = new_value`). If you try to modify it using an object instance (e.g., `self.static_variable = new_value`), you will inadvertently create a new instance variable that "shadows" the class variable for that specific object, leaving the actual class variable unchanged.
3.  **"Can an instance method access a static variable?"**
    * *Answer:* Yes. An instance method can read a static variable either by using `self.static_variable` (as long as it hasn't been shadowed by an instance variable of the same name) or, more safely and preferably, by using `ClassName.static_variable`.
4.  **"Where are instance variables and static variables stored in Python's memory?"**
    * *Answer:* Instance variables are stored in the instance's private dictionary, accessible via `obj.__dict__`. Static variables are stored in the class's dictionary, accessible via `ClassName.__dict__`.